# Healthcare Data Engineering — Silver Layer Transformation

## Overview
This notebook demonstrates a **Bronze-to-Silver data transformation workflow** for a healthcare patient dataset using **PySpark** and **Delta Lake**.

### Pipeline Objectives:
- Load raw patient records from the **Bronze layer**
- Audit dataset schema and run **Null-value quality checks**
- Enforce entity integrity by removing duplicate and null `patient_id` values
- Standardize text fields (`first_name`, `last_name`, `city`, `gender`)
- Cast string dates into native `DateType` and derive patient `age`
- Validate data quality for logical anomalies (e.g., age bounds)
- Append audit trail metadata (`processed_timestamp`)
- Write the normalized dataset to the **Silver layer** as a Delta table

---

## 1. Environment & Table Configuration
Define source and target Delta Lake table paths. Using constants allows easy adaptation across Databricks or local environments.

In [ ]:
# Define table locations in Unity Catalog / Hive Metastore
BRONZE_TABLE = "workspace.healthcare.bronze_patients"
SILVER_TABLE = "workspace.healthcare.silver_patients"

print(f"Configured Source Table: {BRONZE_TABLE}")
print(f"Configured Target Table: {SILVER_TABLE}")

## 2. Read Raw Bronze Dataset
Extract raw patient data from the Bronze Delta table into a PySpark DataFrame.

In [ ]:
# Load raw bronze data table
patients_bronze = spark.table(BRONZE_TABLE)

# Display sample records for inspection
display(patients_bronze)

## 3. Schema Inspection
Print the DataFrame schema to inspect initial column data types before applying transformations.

In [ ]:
# Inspect column structure and data types
patients_bronze.printSchema()

## 4. Import PySpark Transformation Functions
Import essential PySpark SQL functions needed for data cleansing, casting, and feature engineering.

In [ ]:
from pyspark.sql.functions import (
    col,                # References DataFrame columns
    sum,                # Aggregates null counts
    when,               # Conditional logic for null checks
    trim,               # Cleans leading/trailing whitespaces
    initcap,            # Converts strings to Title Case
    upper,              # Converts strings to Uppercase
    to_date,            # Parses string formats into DateType
    year,               # Extracts year integer from date
    current_date,       # Returns current date
    current_timestamp   # Returns current timestamp for tracking
)

## 5. Audit Data Quality: Null-Value Check
Programmatically count null occurrences across all dataset columns using Python list comprehensions and PySpark aggregation.

In [ ]:
# Compute null counts for every column dynamically
null_check = patients_bronze.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in patients_bronze.columns
])

# Output summary matrix showing null counts per column
display(null_check)

## 6. Deduplication Strategy
Remove redundant records that share identical primary identifiers (`patient_id`).

In [ ]:
# Drop duplicate records on primary key identifier
patients_clean = patients_bronze.dropDuplicates(["patient_id"])

# Output record count after deduplication
print("Total unique records remaining:", patients_clean.count())

## 7. Primary Key Validation
Filter out any records where the primary key `patient_id` is null to guarantee entity integrity.

In [ ]:
# Retain records with valid, non-null patient_id
patients_clean = patients_clean.filter(
    col("patient_id").isNotNull()
)

## 8. String Standardizations
Apply string transformations to sanitize free-text attributes:
- `first_name`, `last_name`, `city` $\rightarrow$ Strip whitespaces and set to Title Case.
- `gender` $\rightarrow$ Strip whitespaces and set to Upper Case.

In [ ]:
# Apply string cleaning transformations sequentially
patients_clean = patients_clean \
    .withColumn("first_name", initcap(trim(col("first_name")))) \
    .withColumn("last_name", initcap(trim(col("last_name")))) \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn("gender", upper(trim(col("gender"))))

## 9. Date Type Casting
Convert `registration_date` from a generic string into a native PySpark `DateType`.

In [ ]:
# Cast string registration date into standard SQL DateType (YYYY-MM-DD)
patients_clean = patients_clean.withColumn(
    "registration_date",
    to_date(col("registration_date"))
)

## 10. Feature Derivation & Outlier Detection
Derive patient `age` dynamically based on birth year relative to the current execution date, and filter for data anomalies ($age < 0$ or $age > 120$).

In [ ]:
# Calculate dynamic age column from date_of_birth
patients_with_age = patients_clean.withColumn(
    "age",
    year(current_date()) - year(to_date(col("date_of_birth")))
)

# Filter and inspect potential age outliers/anomalies
invalid_age = patients_with_age.filter(
    (col("age") < 0) |
    (col("age") > 120)
)

print("Total invalid age records detected:", invalid_age.count())
display(invalid_age)

## 11. Add Pipeline Lineage Audit Timestamp
Append `processed_timestamp` to track exactly when each Silver record was generated.

In [ ]:
# Add operational processing timestamp
patients_final = patients_with_age.withColumn(
    "processed_timestamp",
    current_timestamp()
)

# Preview finalized DataFrame before persisting
display(patients_final)

## 12. Validate Final Silver Schema
Print the schema of the cleansed dataset to confirm correct column names and data types.

In [ ]:
# Verify target schema types
patients_final.printSchema()

## 13. Persist Cleaned Data to Silver Delta Table
Write the processed DataFrame into the target Silver Delta Lake table using `overwrite` mode.

In [ ]:
# Save transformed data into Silver Delta table
patients_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_TABLE)

## 14. Read Back & Verify Target Table
Query the newly written Silver Delta table directly to verify persistence.

In [ ]:
# Load persisted Delta table for validation
silver_patients = spark.table(SILVER_TABLE)

# Display written Silver table content
display(silver_patients)

## Transformation Summary

| Stage | Operation Applied | PySpark Logic |
|---|---|---|
| **Ingestion** | Read Bronze Delta Table | `spark.table()` |
| **Quality Check** | Null Value Audit | `sum(when(col().isNull(), 1))` |
| **Deduplication** | Remove Duplicate IDs | `.dropDuplicates(["patient_id"])` |
| **Filtering** | Null Key Removal | `.filter(col("patient_id").isNotNull())` |
| **Sanitization** | Trim & Standardize Text | `initcap(trim())`, `upper(trim())` |
| **Casting** | String to DateType | `to_date(col())` |
| **Feature Engineering** | Derive Patient Age | `year(current_date()) - year(...)` |
| **Audit Logging** | Append System Timestamp | `current_timestamp()` |
| **Persistence** | Overwrite Silver Delta Table | `.write.format("delta").saveAsTable()` |

---